# 04 — Alternative spatial sLCC visualization

This notebook is **visualization-only**. It reads the state-level sLCC tables already produced by `03_spatial_lca_tea_slcc.ipynb`.

It reproduces the intended legacy 4 × 2 figure semantics:

- **Rows 1–3:** selected impact-category **ΔsLCC vs market** (`sLCC_green_vs_market_abs`, $/kg).
- **Row 4:** **market-average ΔsLCC change (%) across all six LCIA categories**.
- **Columns:** BC2 (left) and BO2 (right).
- `RdBu_r`, histogram/ECDF equalization, zero-centered diverging color scale, black state borders, and one shared colorbar per row.

No TEA, LCA, damage monetization, SPC, or sLCC calculation is rerun.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

def find_project_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "graphite_sus").is_dir() and (candidate / "result" / "spatial").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphite_sus.plots.spatial_slcc_plotly_panels import (
    CANON_IMPACTS,
    DEFAULT_IMPACT_GROUPS,
    IMPACT_LABEL,
    assert_updated_spatial_outputs,
    impact_group_slug,
    load_slcc_results,
    make_slcc_gap_marketavg_4x2_panel,
    market_average_pct_table,
    save_plotly_panel,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2


## 1. Confirm that notebook 03 used the updated energy-factor workflow

In [2]:
STRICT_UPDATED_MODEL_CHECK = True

run_info = assert_updated_spatial_outputs(
    PROJECT_ROOT,
    strict=STRICT_UPDATED_MODEL_CHECK,
)

display(pd.DataFrame([{
    "manifest": str(run_info.manifest_path.relative_to(PROJECT_ROOT)),
    "social_rate": run_info.social_rate,
    "spc": run_info.spc,
    "damage_scenario": run_info.damage_scenario,
    "electricity_ef_table": run_info.electricity_ef_table,
}]))


,manifest,social_rate,spc,damage_scenario,electricity_ef_table
0,result/spatial/run_manifest.json,0.03,1.1,central,/mnt/g/My Drive/yale/Project-graphite/graphite...


## 2. Figure configuration

In [3]:
LEFT_SCENARIO = "s_c2"   # BC2
RIGHT_SCENARIO = "s_o2"  # BO2

# Plot both notebook-03 market comparisons. Change to ("synthetic",) if desired.
MARKET_TYPES = ("synthetic", "natural")

# Figure 1: Climate / Eutrophication / Respiratory effects
# Figure 2: Acidification / Smog / Freshwater ecotoxicity
IMPACT_GROUPS = DEFAULT_IMPACT_GROUPS

OUTPUT_DIR = PROJECT_ROOT / "result" / "spatial" / "alternative_slcc_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_HTML = True
SAVE_PNG = True
PNG_SCALE = 2.0

display(pd.DataFrame([
    {
        "figure": i + 1,
        "row_1": IMPACT_LABEL[g[0]],
        "row_2": IMPACT_LABEL[g[1]],
        "row_3": IMPACT_LABEL[g[2]],
        "row_4": "Market-average ΔsLCC change across all 6 impacts (%)",
    }
    for i, g in enumerate(IMPACT_GROUPS)
]))


,figure,row_1,row_2,row_3,row_4
0,1,Climate change,Eutrophication,Respiratory effects,Market-average ΔsLCC change across all 6 impac...
1,2,Acidification,Smog,Ecotoxicity: freshwater,Market-average ΔsLCC change across all 6 impac...


## 3. Load notebook-03 sLCC outputs

In [4]:
slcc = {
    market: load_slcc_results(market, project_root=PROJECT_ROOT)
    for market in MARKET_TYPES
}

summary = []
for market, df in slcc.items():
    avg = market_average_pct_table(df, impacts=CANON_IMPACTS)
    summary.append({
        "market": market,
        "rows": len(df),
        "states": df["state"].nunique(),
        "scenarios": df["scenario"].nunique(),
        "impact_categories": df["impact_category"].nunique(),
        "row4_min_pct": avg["market_avg_slcc_change_pct"].min(),
        "row4_mean_pct": avg["market_avg_slcc_change_pct"].mean(),
        "row4_max_pct": avg["market_avg_slcc_change_pct"].max(),
    })

display(pd.DataFrame(summary))


,market,rows,states,scenarios,impact_categories,row4_min_pct,row4_mean_pct,row4_max_pct
0,synthetic,1224,51,4,6,-20.466952,-5.522728,39.329508
1,natural,1224,51,4,6,5.942792,26.939570,88.092385


## 4. Generate the corrected 4 × 2 figures

For each market comparison:

1. Climate change ΔsLCC
2. Eutrophication ΔsLCC
3. Respiratory effects ΔsLCC
4. Market-average ΔsLCC change (%) across all six impacts

and a second panel containing Acidification / Smog / Freshwater ecotoxicity with the same row-4 market-average summary.


In [8]:
generated = []
figures = {}

for market, df in slcc.items():
    for impacts in IMPACT_GROUPS:
        fig = make_slcc_gap_marketavg_4x2_panel(
            df,
            impacts=impacts,
            left_scenario=LEFT_SCENARIO,
            right_scenario=RIGHT_SCENARIO,
            colorscale="RdBu_r",
            width=1100,
            height=1200,
            title_font_size=18,
        )
        key = (market, impacts)
        figures[key] = fig

        base = OUTPUT_DIR / (
            f"slcc_{market}_{impact_group_slug(impacts)}_marketavg_4x2"
        )
        paths = save_plotly_panel(
            fig,
            base,
            write_html=SAVE_HTML,
            write_png=SAVE_PNG,
            scale=PNG_SCALE,
        )
        generated.append({
            "market_type": market,
            "row_1": IMPACT_LABEL[impacts[0]],
            "row_2": IMPACT_LABEL[impacts[1]],
            "row_3": IMPACT_LABEL[impacts[2]],
            "row_4": "Market-average ΔsLCC change (%)",
            "html": str(paths["html"]) if paths["html"] else None,
            "png": str(paths["png"]) if paths["png"] else None,
        })
        fig.show()

output_table = pd.DataFrame(generated)
display(output_table)
output_table.to_csv(OUTPUT_DIR / "generated_marketgap_4x2_panels.csv", index=False)


/tmp/ipykernel_662465/4146165456.py:22: RuntimeWarning:

PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). HTML was still written.



/tmp/ipykernel_662465/4146165456.py:22: RuntimeWarning:

PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). HTML was still written.



/tmp/ipykernel_662465/4146165456.py:22: RuntimeWarning:

PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). HTML was still written.



/tmp/ipykernel_662465/4146165456.py:22: RuntimeWarning:

PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). HTML was still written.



,market_type,row_1,row_2,row_3,row_4,html,png
0,synthetic,Climate change,Eutrophication,Respiratory effects,Market-average ΔsLCC change (%),/mnt/g/My Drive/yale/Project-graphite/graphite...,None
1,synthetic,Acidification,Smog,Ecotoxicity: freshwater,Market-average ΔsLCC change (%),/mnt/g/My Drive/yale/Project-graphite/graphite...,None
2,natural,Climate change,Eutrophication,Respiratory effects,Market-average ΔsLCC change (%),/mnt/g/My Drive/yale/Project-graphite/graphite...,None
3,natural,Acidification,Smog,Ecotoxicity: freshwater,Market-average ΔsLCC change (%),/mnt/g/My Drive/yale/Project-graphite/graphite...,None


## 5. Reference figure matching the supplied example

In [6]:
REFERENCE_MARKET = "synthetic"
REFERENCE_IMPACTS = (
    "climate change",
    "eutrophication",
    "particulate matter formation",
)

reference_fig = make_slcc_gap_marketavg_4x2_panel(
    slcc[REFERENCE_MARKET],
    impacts=REFERENCE_IMPACTS,
    left_scenario="s_c2",
    right_scenario="s_o2",
    colorscale="RdBu_r",
    width=1100,
    height=1200,
    title_font_size=18,
)
reference_fig.show()
